[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhar174/research_corpus_organizer/blob/main/rag_pdf_system.ipynb)

# RAG PDF Research Corpus System

**Version:** 1.0  
**Date:** 2025-11-21  
**Author/Maintainer:** Research Corpus Organizer  
**Based on:** RAG_PDF_System_Spec_v2.1

---

## Overview

This notebook implements a comprehensive system for processing academic PDF research papers using:
- **LangGraph** workflows for orchestration
- **GPT-5.1 Thinking** for summarization and classification
- **FAISS** for vector indexing and RAG queries
- **3-tier hierarchical topic taxonomy** for organization

### System Capabilities

1. **Ingest** PDFs from Google Drive
2. **Parse and chunk** documents with section awareness
3. **Extract metadata** from arXiv/CrossRef/PDF sources
4. **Generate summaries** using advanced LLMs
5. **Build topic taxonomy** through clustering
6. **Classify papers** into hierarchical topics
7. **Enable RAG queries** for corpus exploration

---

## Phase 0: Environment Setup and Configuration

This phase establishes the notebook environment, installs dependencies, and configures the system.

### Step 0.2: Environment Inspection

First, let's verify the runtime environment meets our requirements.

In [ ]:
# Check Python version (require 3.10+)
import sys
print(f"Python version: {sys.version}")
print(f"Version info: {sys.version_info}")

if sys.version_info >= (3, 10):
    print("✓ Python 3.10+ requirement met")
else:
    print("✗ WARNING: Python 3.10+ required")

In [ ]:
# Check GPU/CPU availability
import os

# Try to check for GPU
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"  CUDA version: {torch.version.cuda}")
    else:
        print("○ No GPU available (using CPU)")
except ImportError:
    print("○ PyTorch not installed yet (will check GPU after installation)")

# Display basic system info
import platform
print(f"\nSystem: {platform.system()} {platform.release()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")

In [ ]:
# Display runtime information
import os
try:
    import psutil
    
    # Memory information
    memory = psutil.virtual_memory()
    print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
    print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
    print(f"Used RAM: {memory.used / (1024**3):.2f} GB ({memory.percent}%)")
    
    # Disk space
    disk = psutil.disk_usage('/')
    print(f"\nTotal Disk: {disk.total / (1024**3):.2f} GB")
    print(f"Available Disk: {disk.free / (1024**3):.2f} GB")
    print(f"Used Disk: {disk.used / (1024**3):.2f} GB ({disk.percent}%)")
    
    # CPU information
    print(f"\nCPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
except ImportError:
    print("psutil not installed yet - will be available after dependency installation")

### Step 0.3: Install Dependencies

Install all required packages for the RAG PDF Research Corpus System.

**Note:** After installation, you may need to restart the runtime if prompted.

In [ ]:
# Install all required dependencies
# This cell may take several minutes to complete

import sys

# Core dependencies with specific versions
dependencies = [
    "openai>=1.3.0",           # GPT-5.1 support
    "langgraph>=0.0.30",       # Workflow orchestration
    "langchain>=0.1.0",        # LangChain integration
    "pymupdf>=1.23.0",         # PDF parsing (fitz)
    "faiss-cpu>=1.7.4",        # Vector indexing (CPU version)
    "scikit-learn>=1.3.0",     # Clustering algorithms
    "hdbscan>=0.8.33",         # Density-based clustering
    "pandas>=2.0.0",           # Data handling
    "numpy>=1.24.0",           # Numerical operations
    "tqdm>=4.65.0",            # Progress bars
    "matplotlib>=3.7.0",       # Visualization
    "seaborn>=0.12.0",         # Statistical visualization
    "python-dateutil>=2.8.2",  # Date parsing
    "requests>=2.31.0",        # HTTP requests for APIs
    "pytesseract>=0.3.10",     # OCR (optional)
    "Pillow>=10.0.0",          # Image processing for OCR
    "pydantic>=2.0.0",         # Data validation
    "psutil>=5.9.0",           # System utilities
]

print("Installing dependencies...")
print("=" * 60)

for dep in dependencies:
    print(f"Installing {dep}...")
    !pip install -q {dep}

print("=" * 60)
print("✓ All dependencies installed successfully!")
print("\n⚠ If you see any warnings about restarting the runtime, please do so now.")
print("   After restart, skip this installation cell and continue with imports.")

### Step 0.4: Import Statements

Import all required libraries and verify they load successfully.

In [ ]:
# Standard library imports
import os
import sys
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime, date
from typing import Optional, Dict, List, Literal, Any, TypedDict
from dataclasses import dataclass, field

# Third-party imports - Core
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Third-party imports - Data validation
from pydantic import BaseModel, Field, field_validator

# Third-party imports - PDF processing
try:
    import fitz  # PyMuPDF
    print("✓ PyMuPDF (fitz) imported successfully")
except ImportError as e:
    print(f"✗ Error importing PyMuPDF: {e}")
    fitz = None

# Third-party imports - Vector store
try:
    import faiss
    print("✓ FAISS imported successfully")
except ImportError as e:
    print(f"✗ Error importing FAISS: {e}")
    faiss = None

# Third-party imports - ML/Clustering
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
try:
    import hdbscan
    print("✓ HDBSCAN imported successfully")
except ImportError:
    print("○ HDBSCAN not available (optional)")
    hdbscan = None

# Third-party imports - Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Third-party imports - API and utilities
import requests
from dateutil import parser as date_parser

# Third-party imports - OpenAI
try:
    from openai import OpenAI
    print("✓ OpenAI SDK imported successfully")
except ImportError as e:
    print(f"✗ Error importing OpenAI: {e}")
    OpenAI = None

# Third-party imports - LangGraph
try:
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver
    print("✓ LangGraph imported successfully")
except ImportError as e:
    print(f"✗ Error importing LangGraph: {e}")
    StateGraph = None

# Third-party imports - OCR (optional)
try:
    import pytesseract
    from PIL import Image
    print("✓ OCR libraries imported successfully")
except ImportError:
    print("○ OCR libraries not available (optional)")
    pytesseract = None
    Image = None

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("\n✓ All core imports completed successfully!")

### Step 0.5: Configuration

Define the configuration schema and user-editable configuration.

**Important:** The OpenAI API key should be set via environment variable, never hard-coded.

In [ ]:
# Import the RunConfig from rag_models
# If running locally, ensure rag_models.py is in the same directory
try:
    from rag_models import (
        RunConfig, 
        create_default_config,
        get_openai_client,
        call_gpt5_mini_text,
        call_gpt5_mini_json,
        embed_texts
    )
    print("✓ rag_models imported successfully")
except ImportError as e:
    print(f"✗ Error importing rag_models: {e}")
    print("  Make sure rag_models.py is in the same directory or cloned from the repository")

In [ ]:
# =============================================================================
# USER-EDITABLE CONFIGURATION
# =============================================================================
# Modify the values below to customize the pipeline for your needs.
# All parameters have sensible defaults.

config = RunConfig(
    # ===== Google Drive Settings =====
    drive_folder_path="PDFs",  # Folder in Google Drive containing your PDFs
    
    # ===== OpenAI API Settings =====
    # API key is read from environment variable (never hard-code!)
    openai_api_key_env_var="OPENAI_API_KEY",
    default_text_model="gpt-5-mini",           # Default model for text generation
    default_embedding_model="text-embedding-3-small",  # Default embedding model
    api_timeout_seconds=120,                    # API call timeout
    api_max_batch_size=100,                     # Max batch size for batch API
    
    # ===== Processing Limits =====
    max_papers_per_run=None,     # None = process all; set a number to limit
    max_pages_per_paper=None,    # None = all pages; set a number to limit
    max_chunks_per_paper=100,    # Maximum chunks per paper
    
    # ===== Model Selections =====
    summary_model="gpt-5-mini",         # Model for generating summaries
    taxonomy_model="gpt-5-mini",        # Model for taxonomy generation
    classification_model="gpt-5-mini",  # Model for paper classification
    embedding_model="text-embedding-3-large",  # Model for embeddings
    
    # ===== Reasoning Effort (for GPT thinking models) =====
    summary_reasoning_effort="medium",       # none, low, medium, high
    taxonomy_reasoning_effort="high",        # Higher for complex taxonomy
    classification_reasoning_effort="medium",
    
    # ===== Clustering Parameters =====
    cluster_tier1_target_k=8,    # Target Tier 1 topics (broad areas)
    cluster_tier2_target_k=3,    # Target Tier 2 topics per Tier 1
    cluster_tier3_target_k=2,    # Target Tier 3 topics per Tier 2
    
    # ===== Chunk Size Parameters =====
    chunk_size_chars=1500,       # Target chunk size in characters
    chunk_overlap_chars=200,     # Overlap between chunks
    
    # ===== Token Limits =====
    max_tokens_per_summary=2000,        # Max tokens for summaries
    max_tokens_per_classification=1000, # Max tokens for classification
    
    # ===== Feature Flags =====
    enable_ocr_fallback=False,           # Enable OCR for scanned PDFs
    enable_deep_analysis_pass=False,     # Enable deep analysis (Pass 2)
    taxonomy_approval_required=True,     # Require manual taxonomy approval
    use_tiered_models=False,             # Use cheaper models for bulk tasks
    
    # ===== Budget & Cost Controls =====
    max_cost_per_run=None,              # None = no limit; set $ amount to limit
    cost_warning_threshold=0.8,         # Warn at 80% of budget
    enable_cost_tracking=True,          # Track API costs
    enable_result_caching=True,         # Cache results to avoid duplicates
    batch_api_calls=True,               # Use batch API for 50% discount
)

print("Configuration created successfully!")

In [ ]:
# Display the active configuration
print(config.display_config())

In [ ]:
# =============================================================================
# OpenAI API Key Setup
# =============================================================================
# The API key should be set as an environment variable.
# In Google Colab, you can use:
#   - Secrets (recommended): Use the Colab secrets feature
#   - Environment variable: os.environ['OPENAI_API_KEY'] = 'your-key'
#
# NEVER hard-code your API key in the notebook!

import os

# Check if API key is set
api_key_env = config.openai_api_key_env_var
if os.environ.get(api_key_env):
    print(f"✓ OpenAI API key found in environment variable '{api_key_env}'")
    # Verify by initializing client
    try:
        client = get_openai_client(config)
        print("✓ OpenAI client initialized successfully")
    except Exception as e:
        print(f"✗ Error initializing OpenAI client: {e}")
else:
    print(f"⚠ OpenAI API key not found in environment variable '{api_key_env}'")
    print("  Set your API key using one of these methods:")
    print("  1. Colab Secrets (recommended for Colab)")
    print("  2. os.environ['OPENAI_API_KEY'] = 'your-key-here'")
    print("  3. Export in terminal: export OPENAI_API_KEY='your-key-here'")

In [ ]:
# =============================================================================
# Configuration Validation
# =============================================================================
# Validate the configuration parameters

def validate_config(cfg: RunConfig) -> dict:
    """
    Validate configuration parameters and return validation results.
    
    Args:
        cfg: RunConfig instance to validate
        
    Returns:
        Dictionary with validation results
    """
    issues = []
    warnings = []
    
    # Check drive folder path
    if not cfg.drive_folder_path:
        issues.append("drive_folder_path is empty")
    
    # Check model names
    valid_text_models = ["gpt-5-mini", "gpt-5", "gpt-4", "gpt-4-turbo", "o4-mini", "o4"]
    if cfg.summary_model not in valid_text_models:
        warnings.append(f"summary_model '{cfg.summary_model}' is not a recognized model")
    
    valid_embedding_models = ["text-embedding-3-small", "text-embedding-3-large"]
    if cfg.embedding_model not in valid_embedding_models:
        warnings.append(f"embedding_model '{cfg.embedding_model}' is not a recognized model")
    
    # Check chunk parameters
    if cfg.chunk_overlap_chars >= cfg.chunk_size_chars:
        issues.append("chunk_overlap_chars must be less than chunk_size_chars")
    
    # Check budget
    if cfg.max_cost_per_run is not None and cfg.max_cost_per_run <= 0:
        issues.append("max_cost_per_run must be positive or None")
    
    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "warnings": warnings
    }

# Run validation
validation_result = validate_config(config)

if validation_result["valid"]:
    print("✓ Configuration is valid")
else:
    print("✗ Configuration has issues:")
    for issue in validation_result["issues"]:
        print(f"  - {issue}")

if validation_result["warnings"]:
    print("\n⚠ Warnings:")
    for warning in validation_result["warnings"]:
        print(f"  - {warning}")

---

## Phase 1: Data Models and Schema Definitions

This phase defines all core data structures used throughout the pipeline.
The models are implemented in `rag_models.py` and include:

- **PaperRecord**: Comprehensive record for each research paper
- **PaperChunk**: Text chunks for RAG indexing
- **TopicHierarchy**: 3-tier topic taxonomy
- **GraphState**: LangGraph workflow state
- **Helper classes**: Metadata extraction, statistics, error handling

In [ ]:
# Import all data models from rag_models
from rag_models import (
    # Core Models
    PaperRecord,
    PaperChunk,
    TopicNode,
    TopicHierarchy,
    GraphState,
    
    # State Management
    StateManager,
    
    # Helper Classes
    MetadataExtractor,
    StatisticsTracker,
    ErrorHandler,
    IDGenerator,
    
    # Cost Tracking
    CostTracker,
    
    # Utility Functions
    validate_paper_record,
)

print("✓ All data models imported successfully")
print("\nAvailable models:")
print("  - PaperRecord: Paper metadata and processing status")
print("  - PaperChunk: Text chunks for RAG indexing")
print("  - TopicHierarchy: 3-tier topic taxonomy")
print("  - GraphState: LangGraph workflow state")
print("  - StateManager: State management utilities")
print("  - CostTracker: API cost tracking")

In [ ]:
# Initialize the GraphState with configuration
state = StateManager.create_initial_state(config)

print("✓ Initial GraphState created")
print(f"\nState fields:")
print(f"  - current_phase: {state['current_phase']}")
print(f"  - papers: {len(state['papers'])} papers")
print(f"  - chunks: {len(state['chunks'])} paper chunk sets")
print(f"  - topic_hierarchy: {'set' if state['topic_hierarchy'] else 'not set'}")
print(f"  - taxonomy_approved: {state['taxonomy_approved']}")

---

## Next Steps

With Phase 0 (Environment Setup) and Phase 1 (Data Models) complete, the system is ready for:

- **Phase 2**: Google Drive Integration
- **Phase 3**: PDF Parsing and Chunking
- **Phase 4**: Metadata Extraction
- **Phase 5**: Embedding Generation and FAISS Index
- And subsequent phases...

Refer to `FINAL_NOTEBOOK_ACTION_PLAN.md` for the complete implementation roadmap.